# Week 1 — Python, Data Exploration and First Models

**Domain:** Education / student performance. **Question:** Which study and learning-context factors are associated with exam scores, and how reliably can we predict a score?

This notebook covers all six Week 1 tasks. It uses the public Student Performance Factors dataset (6,607 rows, 20 columns). The dataset's sampling process and population are not fully documented, so results are exploratory and must not be interpreted as causal or used to make decisions about individual students.

**Run:** Google Colab or Jupyter. Run all cells from top to bottom. The GPU check is optional in Week 1; choose a T4 runtime if available.

## 1. Python basics

The next cell demonstrates the required built-in types, list, dictionary, loop, conditional and a function that returns a value.

In [ ]:
student_count = 6607          # int
average_score = 67.2          # float
domain = 'education'          # str
is_regression_task = True     # bool

study_topics = ['Python', 'Pandas', 'Statistics', 'Machine Learning', 'Ethics']
print('Third topic:', study_topics[2])

student_profile = {
    'student_id': 'S-001', 'attendance': 92,
    'study_hours': 18, 'support_needed': False
}
print('Profile fields:', len(student_profile))

for topic in study_topics:
    print('Learning:', topic)

if student_profile['attendance'] < 75:
    attendance_note = 'Attendance support may be useful'
elif student_profile['attendance'] < 90:
    attendance_note = 'Attendance is moderate'
else:
    attendance_note = 'Attendance is high'
print(attendance_note)

def score_gap(actual, predicted):
    return actual - predicted

example_gap = score_gap(82, 78)
print('Returned score gap:', example_gap, '| doubled:', example_gap * 2)

## 2. Load and inspect a real dataset

The CSV is hosted in a public GitHub dataset mirror. The original dataset is commonly attributed to the Kaggle *Student Performance Factors* dataset. The mirror is used only as a convenient download location; check the original source and licence before redistribution.

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = 'https://raw.githubusercontent.com/omaarrx/Student-Score-Prediction/main/StudentPerformanceFactors.csv'
DATA_PATH = 'StudentPerformanceFactors.csv'
if not os.path.exists(DATA_PATH):
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
df_raw = pd.read_csv(DATA_PATH)
print('Shape:', df_raw.shape)
print('\nTypes:\n', df_raw.dtypes)
print('\nMissing values:\n', df_raw.isna().sum())
display(df_raw.head())
assert len(df_raw) >= 500 and df_raw.shape[1] >= 5

In [ ]:
numeric_cols = df_raw.select_dtypes(include=np.number).columns
summary = df_raw[numeric_cols].agg(['min','max','mean','median']).T
summary.columns = ['minimum','maximum','mean','median']
display(summary.round(2))
text_cols = df_raw.select_dtypes(exclude=np.number).columns.tolist()
if text_cols:
    col = text_cols[0]
    print('Distinct values in', col, ':', df_raw[col].nunique())
    display(df_raw[col].value_counts().head(5))

# Column selection and a meaningful filter
print(df_raw['Hours_Studied'].head())
display(df_raw[['Hours_Studied','Attendance']].head())
display(df_raw.loc[df_raw['Attendance'] < 75, ['Attendance','Exam_Score']].head())

In [ ]:
values = df_raw['Hours_Studied'].to_numpy()
start = time.perf_counter(); loop_sum = sum(float(v) for v in values); loop_seconds = time.perf_counter()-start
start = time.perf_counter(); vector_sum = values.sum(); vector_seconds = time.perf_counter()-start
print({'loop_sum':loop_sum,'numpy_sum':float(vector_sum),'loop_seconds':loop_seconds,'numpy_seconds':vector_seconds})
print('Speed ratio (loop / NumPy):', loop_seconds / max(vector_seconds, 1e-12))

## 3. Cleaning, outliers, charts and ethics

Missing categorical values are filled with the most frequent category. This preserves rows, but mode imputation can under-represent rare groups. Numeric columns are left unchanged unless missing; the median is used where needed. Outliers are inspected and retained unless a value is demonstrably invalid: an unusual score is not automatically an error.

In [ ]:
df = df_raw.copy()
missing_notes = {}
for col in df.columns:
    n_missing = int(df[col].isna().sum())
    if n_missing:
        if pd.api.types.is_numeric_dtype(df[col]):
            fill_value = df[col].median()
            df[col] = df[col].fillna(fill_value)
            missing_notes[col] = f'Filled {n_missing} missing numeric values with median {fill_value:.2f}; robust to skew.'
        else:
            fill_value = df[col].mode(dropna=True).iloc[0]
            df[col] = df[col].fillna(fill_value)
            missing_notes[col] = f'Filled {n_missing} missing categorical values with mode {fill_value!r}; preserves rows but may favour majority category.'
print('\n'.join(f'{k}: {v}' for k,v in missing_notes.items()) or 'No missing values found.')
print('Remaining missing:', int(df.isna().sum().sum()))

col='Exam_Score'; q1,q3=df[col].quantile([.25,.75]); iqr=q3-q1
lo,hi=q1-1.5*iqr,q3+1.5*iqr
outliers=df.loc[(df[col]<lo)|(df[col]>hi), col]
print(f'{col}: IQR bounds {lo:.1f}..{hi:.1f}; {len(outliers)} potential outliers; observed range {df[col].min()}..{df[col].max()}')
print('Decision: inspect extreme values; retain plausible records and do not cap scores automatically.')

In [ ]:
fig, ax = plt.subplots(figsize=(7,4)); ax.hist(df['Exam_Score'], bins=25); ax.set(title='Distribution of exam scores', xlabel='Exam score', ylabel='Number of records'); plt.show()
fig, ax = plt.subplots(figsize=(7,4)); df['Parental_Involvement'].value_counts().plot(kind='bar', ax=ax); ax.set(title='Records by parental involvement', xlabel='Parental involvement', ylabel='Number of records'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7,4)); ax.scatter(df['Hours_Studied'], df['Exam_Score'], alpha=.35); ax.set(title='Study hours and exam score', xlabel='Hours studied', ylabel='Exam score'); plt.show()
fig, ax = plt.subplots(figsize=(7,4)); df.groupby('School_Type')['Exam_Score'].mean().plot(kind='bar', ax=ax); ax.set(title='Mean exam score by school type', xlabel='School type', ylabel='Mean exam score'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7,4)); df.groupby(pd.cut(df['Attendance'], bins=[0,60,70,80,90,100], include_lowest=True))['Exam_Score'].mean().plot(kind='bar', ax=ax); ax.set(title='Mean exam score by attendance band', xlabel='Attendance band', ylabel='Mean exam score'); plt.tight_layout(); plt.show()

### Findings and data ethics

After running the charts, record five observations supported by the displayed values. Do not claim causation from correlations. The dataset documentation does not fully establish who was sampled or how the records were collected; therefore representativeness is unknown. Demographic and disability-related attributes could create unfair treatment if used to allocate opportunities. For this baseline, exclude Gender and Learning_Disabilities from predictors, report subgroup performance where appropriate, and never use the model to make high-stakes decisions about students.

In [ ]:
findings = [
    f'Mean exam score: {df.Exam_Score.mean():.2f}; median: {df.Exam_Score.median():.2f}.',
    f'Mean attendance: {df.Attendance.mean():.2f}; correlation with exam score: {df.Attendance.corr(df.Exam_Score):.3f}.',
    f'Mean score by school type: {df.groupby("School_Type").Exam_Score.mean().round(2).to_dict()}.',
    f'Mean score by parental involvement: {df.groupby("Parental_Involvement").Exam_Score.mean().round(2).to_dict()}.',
    f'Mean score by study-hours quartile: {df.assign(study_band=pd.qcut(df.Hours_Studied, 4, duplicates="drop")).groupby("study_band", observed=True).Exam_Score.mean().round(2).to_dict()}.'
]
print('\n'.join(f'{i+1}. {x}' for i,x in enumerate(findings)))

## 4. First supervised model — regression

Predict `Exam_Score` using study and learning-context fields. This is a regression task. Exclude the target and identifiers (if any) from features. A held-out 20% test set is used throughout Weeks 1 and 2.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

TARGET='Exam_Score'
EXCLUDED=['Gender','Learning_Disabilities']
X=df.drop(columns=[TARGET]+EXCLUDED, errors='ignore')
y=df[TARGET].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=42)
num_features=X.select_dtypes(include=np.number).columns.tolist()
cat_features=X.select_dtypes(exclude=np.number).columns.tolist()
preprocess=ColumnTransformer([
 ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_features),
 ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_features)
])
models={
 'Dummy mean':DummyRegressor(strategy='mean'),
 'Linear regression':LinearRegression(),
 'Decision tree depth=5':DecisionTreeRegressor(max_depth=5,random_state=42),
 'Random forest':RandomForestRegressor(n_estimators=200,min_samples_leaf=2,random_state=42,n_jobs=-1)
}
trained={}; rows=[]
for name, estimator in models.items():
    pipe=Pipeline([('preprocess',preprocess),('model',estimator)])
    pipe.fit(X_train,y_train); pred=pipe.predict(X_test); trained[name]=pipe
    rows.append({'model':name,'MAE':mean_absolute_error(y_test,pred),'RMSE':mean_squared_error(y_test,pred)**.5,'R2':r2_score(y_test,pred)})
reg_results=pd.DataFrame(rows).sort_values('RMSE')
display(reg_results.round(3))
print('Train R2 / test R2, depth-unlimited tree (overfit demonstration):')
greedy=Pipeline([('preprocess',preprocess),('model',DecisionTreeRegressor(random_state=42))]).fit(X_train,y_train)
print(round(greedy.score(X_train,y_train),3), round(greedy.score(X_test,y_test),3))

## 5. Evaluation beyond accuracy

The main task is regression, so MAE/RMSE/R² are appropriate. To practise classification metrics, define a transparent secondary label: `needs_support = Exam_Score < 70`. This is a learning exercise, not a validated educational threshold. We compare three classifiers with a leakage-safe preprocessing pipeline.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold

Xc=df.drop(columns=[TARGET]+EXCLUDED, errors='ignore')
yc=(df[TARGET] < 70).astype(int)  # 1 = below-70 score band; illustrative only
Xc_train,Xc_test,yc_train,yc_test=train_test_split(Xc,yc,test_size=.2,random_state=42,stratify=yc)
numc=Xc.select_dtypes(include=np.number).columns.tolist(); catc=Xc.select_dtypes(exclude=np.number).columns.tolist()
prep_c=ColumnTransformer([
 ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numc),
 ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),catc)
])
classifiers={'Logistic regression':LogisticRegression(max_iter=2000),'Decision tree':DecisionTreeRegressor(max_depth=6,random_state=42),'Random forest':RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1)}
classifiers['Decision tree']=__import__('sklearn.tree').tree.DecisionTreeClassifier(max_depth=6,random_state=42)
class_rows=[]; fitted_classifiers={}
for name, clf in classifiers.items():
    p=Pipeline([('prep',prep_c),('model',clf)]).fit(Xc_train,yc_train)
    pred=p.predict(Xc_test); fitted_classifiers[name]=p
    class_rows.append({'model':name,'accuracy':accuracy_score(yc_test,pred),'precision':precision_score(yc_test,pred,zero_division=0),'recall':recall_score(yc_test,pred,zero_division=0),'F1':f1_score(yc_test,pred,zero_division=0)})
    print('\n',name,'\n',confusion_matrix(yc_test,pred),'\n',classification_report(yc_test,pred,zero_division=0))
display(pd.DataFrame(class_rows).sort_values('F1',ascending=False).round(3))

best_name=max(class_rows,key=lambda r:r['F1'])['model']; best=fitted_classifiers[best_name]
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_scores=cross_val_score(best,Xc,yc,cv=cv,scoring='f1')
print('5-fold F1:',np.round(cv_scores,3),'mean=',round(cv_scores.mean(),3),'std=',round(cv_scores.std(),3))
print('Domain error trade-off: false negatives miss students in the illustrative below-70 band; false positives may incorrectly flag students. Any real intervention requires human review and a validated threshold.')

## 6. Unsupervised learning — k-means

Cluster numeric study-context variables without using the exam-score label. Standardise features first because k-means uses distances. Cluster labels are descriptive groups, not natural categories or diagnoses.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
cluster_features=['Hours_Studied','Attendance','Sleep_Hours','Previous_Scores','Tutoring_Sessions','Physical_Activity']
X_cluster=df[cluster_features].copy()
scaler=StandardScaler(); Z=scaler.fit_transform(X_cluster)
k3=KMeans(n_clusters=3,n_init=20,random_state=42)
clusters=k3.fit_predict(Z)
plt.figure(figsize=(6,4)); plt.scatter(Z[:,0],Z[:,1],c=clusters,s=10,alpha=.45); plt.xlabel('Scaled study hours'); plt.ylabel('Scaled attendance'); plt.title('Student learning-context clusters (k=3)'); plt.show()
inertias=[]
for k in range(2,9):
    inertias.append(KMeans(n_clusters=k,n_init=20,random_state=42).fit(Z).inertia_)
plt.figure(figsize=(6,4)); plt.plot(range(2,9),inertias,marker='o'); plt.xlabel('Number of clusters (k)'); plt.ylabel('Inertia'); plt.title('Elbow curve'); plt.show()
df_cluster=df.copy(); df_cluster['cluster']=clusters
display(df_cluster.groupby('cluster')[cluster_features+['Exam_Score']].mean().round(2))
print('Inspect the means above and name clusters based on their profiles. The labels are provisional and should not be used to rank students.')

## Reflection and reproducibility

Before submission, run **Runtime → Restart and run all**. Record five specific findings from your actual run, explain any differences from expected patterns, and write a short note about limitations. Never present correlation as causation.

In [ ]:
print('Dataset rows/columns:',df.shape)
print('Train/test sizes:',len(X_train),len(X_test))
print('Best regression model by RMSE:',reg_results.iloc[0]['model'])
print('Best classification model by F1:',best_name)
print('K-means cluster sizes:',pd.Series(clusters).value_counts().sort_index().to_dict())
print('GPU available (optional for Week 1):')
try:
    import torch
    print(torch.cuda.is_available())
except ImportError:
    print('PyTorch is not installed; GPU check skipped.')